In [ ]:
import pandas as pd
import os

# ====================== 参数设置 ======================
typhoon_sid = "2022099S13088"                # 马库斯台风
save_dir = r"C:\Users\changan\Desktop\Hankelformer-new\dataset\typhoon"
os.makedirs(save_dir, exist_ok=True)

# ====================== 核心：Parquet 远程读取 ======================
# IBTrACS v04r01 提供了按海盆分组的 Parquet 文件，Pandas 可直接读取并下压过滤条件
# 需要先知道该风暴属于哪个海盆。这里用一个简单的一行方案：
list_url = "https://www.ncei.noaa.gov/data/international-best-track-archive-for-climate-stewardship-ibtracs/v04r01/access/csv/ibtracs.ALL.list.v04r01.csv"
df_list = pd.read_csv(list_url, usecols=['SID', 'BASIN'], dtype=str)
basin = df_list[df_list['SID'] == typhoon_sid]['BASIN'].values[0]
print(f"马库斯台风所在海盆: {basin}")

# 构建对应海盆的 Parquet 文件地址
parquet_url = f"https://www.ncei.noaa.gov/data/international-best-track-archive-for-climate-stewardship-ibtracs/v04r01/access/parquet/ibtracs.{basin}.v04r01.parquet"

# pandas 读取时应用过滤，只加载该风暴的数据，所有其他风暴不会进入内存
df = pd.read_parquet(parquet_url, filters=[("SID", "==", typhoon_sid)])

if df.empty:
    # 如果 parquet 未找到，备用：从总 CSV 分块读取
    all_csv_url = "https://www.ncei.noaa.gov/data/international-best-track-archive-for-climate-stewardship-ibtracs/v04r01/access/csv/ibtracs.ALL.v04r01.csv"
    df_iter = pd.read_csv(all_csv_url, chunksize=500000)
    df = pd.concat(chunk[chunk['SID'] == typhoon_sid] for chunk in df_iter)
    del df_iter

# ====================== 保存数据 ======================
print(f"共获取 {len(df)} 个观测点")
df.to_csv(os.path.join(save_dir, "marcus_track.csv"), index=False)

# 保留核心列
core_cols = ['time', 'lat', 'lon', 'wmo_wind', 'wmo_pres']
df[core_cols].to_csv(os.path.join(save_dir, "marcus_latlon.csv"), index=False)

print("✅ 数据已保存，内存占用极小")